In [1]:
#مرحلة تجهيز البيانات قبل التدريب 
import os
import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

train_df = pd.read_csv("artifacts/train.csv")
val_df = pd.read_csv("artifacts/val.csv")
test_df = pd.read_csv("artifacts/test.csv")

drop_cols = [                      # بيانات لا نستخدمها  كميزات موجودة ضمن قائمة  نقوم بتجاهلها اثناء عملية feature enginnering
    "is_late",                     # is_late  نتجاهله لانه يمثل المخرجات ولا يجوز وضعه ضمن المدخلات 
    "order_id",                     # معرفت=ات فريدة تستهلك ذاكرة  وليست ذات معنى تنبؤي 
    "customer_id",
    "customer_city",                 # تحتوي على العديد من الفيم المختلفة  وهذا يسبب High cardinality
    "order_delivered_customer_date",  # is _late  = delivered > estimated   وهذا له علاقة بالخرج وليس الدخل  واذا اصبح دخل هنا يحدث تسريب للبيانات  
    "order_estimated_delivery_date",
    "order_purchase_timestamp",
]

X_train = train_df.drop(columns=drop_cols, errors="ignore")  # حذف الاعمدة السابقة من بيانات التدريب وفي حال عدم وجود ااعمدة اخرى قم بتجاهلها ولا تظهر خطأ
y_train = train_df["is_late"]

X_val = val_df.drop(columns=drop_cols, errors="ignore")
y_val = val_df["is_late"]

X_test = test_df.drop(columns=drop_cols, errors="ignore")
y_test = test_df["is_late"]

print(f"Train Feature shape : {X_train.shape}")

Train Feature shape : (67530, 9)


In [2]:
# تحديد الأعمدة الرقمية والفئوية
num_cols = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist() #اختيار الاعمدة الرقمية ووضعا في قائمة
cat_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist() # اختيار الاعمدة الفئوية ووضعا في قائمة 


num_pipeline = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="median")), # استبدال قيم nan  يالوسيط meedin
        ("scaler", StandardScaler()),         # توحيد المقاييس
    ]
)

 # بايبلاين الفئات (محدد بـ max_categories لتجنب تضخم الذاكرة)
cat_pipeline = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="most_frequent")),  # استدال القيم الناقصة بالاكثر تكرار
        
        ( "encoder",OneHotEncoder(handle_unknown="ignore", sparse_output=False, max_categories=30)), # تحويل الرموز الى ارقام واذا ظهرت فئة جديدة غير مرمزة لا توقف البرنامج 
    ]   # ظهرت مشكلة هنا بخصوص الذاكرة في عدد الفئات فقمت بتحديد عدد الفئات الناتجة  الى 30 فئة مع التعامل مع الفئات الاقل تكرار لتقليل عدد الاعمدة الناتجة 
)

preprocessor = ColumnTransformer(                 # دمج التحويلات 
    transformers=[
        ("num", num_pipeline, num_cols),
        ("cat", cat_pipeline, cat_cols),
    ]
)

C:\Users\HADI\AppData\Local\Temp\ipykernel_2152\4012552768.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist() # اختيار الاعمدة الفئوية ووضعا في قائمة


In [ ]:

X_train_transformed = preprocessor.fit_transform(X_train) # hg preprocessor يتعلم من بينات التدريب ويطبق ما تعلمه على بيانات التدريب 


X_val_transformed = preprocessor.transform(X_val)  # استخدام ال preprocessor  في معالجة بيانات الاختبار والتحقق 
X_test_transformed = preprocessor.transform(X_test)


encoded_cat_cols = (
    preprocessor.named_transformers_["cat"]        #  cat  بال transformerاعطاء اسم ال  
    .named_steps["encoder"]                        # encoded اختيار قسم  
    .get_feature_names_out(cat_cols)                # اعطاء اسماء الاعمدة الخاصة بالميزات  الفئوية 
)
all_feature_names = num_cols + list(encoded_cat_cols)   # دمج البيانات الرقمية والفئوية ضمن قائمة 


X_train_processed = pd.DataFrame(         # data frame  تحويل بيانات التدريب الى 
    X_train_transformed, columns=all_feature_names
)
X_val_processed = pd.DataFrame(X_val_transformed, columns=all_feature_names)
X_test_processed = pd.DataFrame(X_test_transformed, columns=all_feature_names)

print(
    f"Size of train data after processing {X_train_processed.shape}"
)

Size of train data after processing (67530, 94)


In [ ]:
#  is_ lateاضافة الهدف  
X_train_processed["is_late"] = y_train.values       #  index ويأخذ الفيم فقط بغض النظر عن ال  y_train ونعطيه قيمه الهدف الموجودة في  is_late انشاء عمود 
X_val_processed["is_late"] = y_val.values
X_test_processed["is_late"] = y_test.values

os.makedirs("artifacts", exist_ok=True)


X_train_processed.to_csv("artifacts/train_fe.csv", index=False)
X_val_processed.to_csv("artifacts/val_fe.csv", index=False)
X_test_processed.to_csv("artifacts/test_fe.csv", index=False)


joblib.dump(preprocessor, "artifacts/preprocessor.joblib")  #  الى ملف preprocessor تحويل الكائن  

print("Complet this ")



Complet this 


In [5]:
import joblib
model = joblib.load("artifacts/preprocessor.joblib")
print(f" type of model is : {type(model)}")   # random forest
print(model)


 type of model is : <class 'sklearn.compose._column_transformer.ColumnTransformer'>
ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['total_items', 'total_price', 'total_freight',
                                  'total_payment', 'max_installments']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                max_categories=30,
                                                                spars